# Wavelet Roughness — Bader/Helmholtz & Sethares

`comsar.WaveletRoughness` estimates the spectrum of every analysis frame with a
Gaussian-windowed single-frequency correlation (a **Gabor / wavelet**
transform) evaluated on a *continuous* frequency grid, and locates the spectral
peaks with sub-grid precision. This yields much more exact **partial
frequencies** than FFT bin spacing.

From the partials of each frame two **roughness** (sensory-dissonance) measures
are computed:

* **Bader/Helmholtz** — pair roughness peaking at a beating distance of 33 Hz.
* **Sethares** — the Plomp-Levelt / Sethares dissonance curve.

The analysis is ported from R. Bader's *Wavelet* application. Like
`TimbreTrack`, windowing is given in milliseconds, so any sample rate works and
equal durations give equal numbers of frames.

> Requires `bader-comsar` plus `soundfile`.

In [ ]:
from comsar import WaveletRoughness, timbre_player

## Parameters

`threshold` is the key control: only partials whose amplitude is at least this
fraction of the strongest partial in the whole file are kept — **larger
threshold → fewer partials**. `f_min`/`f_max` bound the analysed band, and
`window_ms`/`overlap` set the time resolution (as in `TimbreTrack`).

In [ ]:
WAV = "CHI-109_Luguhu_Hulusheng.wav"

wr = WaveletRoughness(
    window_ms=370.0,   # analysis window length in milliseconds
    overlap=0.5,       # overlap fraction (0 ... <1)
    f_min=50.0,        # lowest analysed frequency (Hz)
    f_max=5000.0,      # highest analysed frequency (Hz, clipped to Nyquist)
    threshold=0.05,    # keep partials >= 5% of the strongest partial
)
wr.verbose = True
result = wr.extract(WAV)          # any sample rate is accepted

## Results

`result.features` holds the two roughness values per frame (indexed by
`time_s`). `result.partials` is a **long-format** table with one row per
detected partial — the number of rows per frame varies.

In [ ]:
result.features.head()

In [ ]:
result.partials.head(8)

## Save as CSV

Two CSV files are written next to the audio: the per-frame roughness track and
the long-format partials. `EXCEL_STYLE = True` writes semicolons + decimal
commas for European Excel (set it to `False` for standard comma CSV; read the
Excel style back with `pd.read_csv(path, sep=";", decimal=",")`).

In [ ]:
EXCEL_STYLE = True
sep, dec = (";", ",") if EXCEL_STYLE else (",", ".")

roughness_csv = WAV.rsplit(".", 1)[0] + "_roughness.csv"
partials_csv  = WAV.rsplit(".", 1)[0] + "_partials.csv"
result.features.to_csv(roughness_csv, sep=sep, decimal=dec)
result.partials.to_csv(partials_csv, sep=sep, decimal=dec, index=False)
print("wrote", roughness_csv, "and", partials_csv)

## Visualisation

The player shows both roughness curves over the grey waveform and — because we
pass `partials=` — an extra **partial-gram** panel: every detected partial as a
dot at its (time, frequency), with the grey level proportional to its
amplitude. Press **Play**; the cursor runs through all panels. Click a legend
entry to hide/show a roughness curve.

In [ ]:
timbre_player(WAV, result.features, partials=result.partials)